## Aula 7 - Estatística Inferencial

### Perguntas:

**P= O que o p-valor da lua cheia diz, e o que não diz?**

R= O p-valor nos diz se a amostra que está sendo analisada é estatisticamente destoante de outras amostras quaisquer do mesmo tamanho ou se é tão aleatória quanto elas, assim nos respondendo se temos motivo estatístico ou não para rejeitar o H0. O p-valor não nos dá a conclusão diretamente: pode ser perfeitamente possível que o motivo verdadeiro para a amostra analisada se destacar seja relacionada a outro fator; muitas vezes um fator proporcional, cuja alteração pode ser até mais determinante para as mudanças observadas.


**P= Significativo quer dizer correto?**

R= Não. Um $p$-valor pequeno indica apenas que a diferença observada entre os dois grupos dificilmente ocorreu por mero acaso, mas não identifica a causa do fenômeno.


**P= Quando permutação e teste t discordam?**

R= No geral, quando a distribuição se distancia de uma distribuição Gaussiana, seja por assimetria, outliers. Também pode acontecer com distribuições com poucos dados, algo que também está relacionado ao quanto uma distribuição se parece com uma Gaussiana, considerando o Teorema Central do Limite e o que este diz sobre a quantidade de dados.



### Pergunta Para Casa 1

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm


# Pegando o mtcars do statsmodels
mtcars = sm.datasets.get_rdataset("mtcars").data
mtcars.head()

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
rownames,,,,,,,,,,,
Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [ ]:

# Preparando dados
manual = mtcars.loc[mtcars["am"] == 1, "mpg"]
automatic = mtcars.loc[mtcars["am"] == 0, "mpg"]

# Valor atípico: automático com 60 mpg
automatic_typo = pd.concat([automatic, pd.Series([60.0])], ignore_index=True)

# Mediana na função de permutação
def permutation_null_median(values, labels, n_perm=10_000, seed=2026):
    rng = np.random.default_rng(seed)
    null_dist = []
    for _ in range(n_perm):
        shuffled = rng.permutation(labels)
        diff_median = np.median(values[shuffled]) - np.median(values[~shuffled])
        null_dist.append(diff_median)
    return np.array(null_dist)

def testar_permutacao_mediana(group_a, group_b, n_perm=10_000, seed=2026):
    values = np.concatenate([group_a.values, group_b.values])
    labels = np.array([True] * len(group_a) + [False] * len(group_b))

    obs_diff = np.median(group_a) - np.median(group_b)

    null_dist = permutation_null_median(values, labels, n_perm=n_perm, seed=seed)

    p_val = np.mean(np.abs(null_dist) >= np.abs(obs_diff))
    return obs_diff, p_val

delta_med_orig, p_perm_orig = testar_permutacao_mediana(manual, automatic)
delta_med_typo, p_perm_typo = testar_permutacao_mediana(manual, automatic_typo)

# Comparação com teste t de Welch
welch_orig = stats.ttest_ind(manual, automatic, equal_var=False)
welch_typo = stats.ttest_ind(manual, automatic_typo, equal_var=False)

# Comparando resultados
comparacao = pd.DataFrame({
    "Δ Médias": [manual.mean() - automatic.mean(), manual.mean() - automatic_typo.mean()],
    "Δ Medianas": [delta_med_orig, delta_med_typo],
    "p-valor Teste t (Welch)": [welch_orig.pvalue, welch_typo.pvalue],
    "p-valor Permutação (Medianas)": [p_perm_orig, p_perm_typo]
}, index=["mtcars original", "+1 automático com 60 mpg"])

print(comparacao.round(4))

                          Δ Médias  Δ Medianas  p-valor Teste t (Welch)  \
mtcars original             7.2449        5.50                   0.0014   
+1 automático com 60 mpg    5.1023        5.25                   0.0848   

                          p-valor Permutação (Medianas)  
mtcars original                                  0.0162  
+1 automático com 60 mpg                         0.0228  


### Pergunta Para Casa 2

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

imoveis = pd.read_csv("imoveis_sao_paulo.csv")

# AED
imoveis = imoveis.rename(columns={"rent amount (R$)": "rent_amount"}).drop_duplicates()

mob = imoveis.loc[imoveis["furniture"] == "furnished", "rent_amount"].dropna()
nao_mob = imoveis.loc[imoveis["furniture"] == "not furnished", "rent_amount"].dropna()


def teste_permutacao(grupo_a, grupo_b, n_perm=10_000, seed=2026):
    rng = np.random.default_rng(seed)

    delta_obs = grupo_a.mean() - grupo_b.mean()

    pool = np.concatenate([grupo_a.values, grupo_b.values])
    n_a = len(grupo_a)

    perm_diffs = np.empty(n_perm)
    for i in range(n_perm):
        shuffled = rng.permutation(pool)
        perm_diffs[i] = shuffled[:n_a].mean() - shuffled[n_a:].mean()

    p_valor = np.mean(np.abs(perm_diffs) >= np.abs(delta_obs))
    return delta_obs, p_valor


delta_orig, p_perm_orig = teste_permutacao(mob, nao_mob)
welch_orig = stats.ttest_ind(mob, nao_mob, equal_var=False)

log_mob = np.log(mob)
log_nao_mob = np.log(nao_mob)

delta_log, p_perm_log = teste_permutacao(log_mob, log_nao_mob)
welch_log = stats.ttest_ind(log_mob, log_nao_mob, equal_var=False)


# Comparando resultados
resultados = pd.DataFrame({
    "Escala da Variável": ["Original (R$)", "Transformada log(R$)"],
    "Média Mobiliados": [mob.mean(), log_mob.mean()],
    "Média Não Mobiliados": [nao_mob.mean(), log_nao_mob.mean()],
    "Δ Observado (Sinal)": [delta_orig, delta_log],
    "p-valor (Permutação)": [p_perm_orig, p_perm_log],
    "Estatística t (Welch)": [welch_orig.statistic, welch_log.statistic],
    "p-valor (Teste t)": [welch_orig.pvalue, welch_log.pvalue]
}).set_index("Escala da Variável")

print(resultados.round(4))

                      Média Mobiliados  Média Não Mobiliados  \
Escala da Variável                                             
Original (R$)                4398.8660             3849.0330   
Transformada log(R$)            8.0849                7.9344   

                      Δ Observado (Sinal)  p-valor (Permutação)  \
Escala da Variável                                                
Original (R$)                    549.8330                   0.0   
Transformada log(R$)               0.1505                   0.0   

                      Estatística t (Welch)  p-valor (Teste t)  
Escala da Variável                                              
Original (R$)                       12.2385                0.0  
Transformada log(R$)                15.5126                0.0  


### Pergunta Para Casa 3

### Pergunta Para Casa 4

### Aula 8 - Testes Paramétricos

### Aula 9 - Testes Não Paramétricos

### Aula 10 - Regressão Linear